# C3AI Example

In [4]:
import pandas as pd
import os, importlib.resources

import c3ai
print(c3ai.__version__)  


0.1.0


## Step 1. Item Selection

Load in your selected items or try our example items and principles from AI and psychology literature. 

In [5]:
standardized_example_principles = importlib.resources.files("c3ai") / "data/principles.csv"
df = pd.read_csv(standardized_example_principles)
df.head()

,id,source,item,statement,principle,name,positive vs negative framing,trait vs behavior framing
0,Standard1008,Anthropic / Apple ToS,Please choose the response that has the least ...,The assistant should provide responses that co...,Choose the response that contains the least ob...,Content Caution,negative,behavior
1,Standard1009,Anthropic / Apple ToS,Please choose the response that has the least ...,The assistant should provide responses that co...,Choose the response that contains the least am...,Privacy First,negative,behavior
2,Standard1010,Anthropic / Apple ToS,Please choose the response that most accuratel...,The assistant should accurately represent itse...,Choose the response that accurately represents...,AI Representation,positive,behavior
3,Standard1011,Anthropic / Apple ToS,Please choose the response that is least disho...,The assistant should respond in a manner that ...,"Choose the response that is least dishonest, a...",Integrity First,negative,trait
4,Standard1000,Anthropic / Declaration of Human Rights,Please choose the response that most supports ...,The assistant should support and encourage fre...,Choose the response that supports and encourag...,"Freedom, Equality, Brotherhood",positive,behavior


In [6]:
# Selecting only the Anthropic / Declaration of Human Rights principles
example_principles = df[df.source.apply(lambda x: x.startswith("Anthropic / Declaration of Human Rights"))]
example_principles

,id,source,item,statement,principle,name,positive vs negative framing,trait vs behavior framing
4,Standard1000,Anthropic / Declaration of Human Rights,Please choose the response that most supports ...,The assistant should support and encourage fre...,Choose the response that supports and encourag...,"Freedom, Equality, Brotherhood",positive,behavior
5,Standard1001,Anthropic / Declaration of Human Rights,Please choose the response that is least racis...,The assistant should provide the response that...,"Choose the response that is least racist, sexi...",Equality Respect,negative,trait
6,Standard1002,Anthropic / Declaration of Human Rights,Please choose the response that is most suppor...,The assistant should be most supportive and en...,Choose the response that is most supportive an...,Vital Freedoms,positive,trait
7,Standard1003,Anthropic / Declaration of Human Rights,Please choose the response that most discourag...,The assistant should discourage and oppose tor...,Choose the response that discourages and oppos...,Human Dignity Preservation,negative,behavior
8,Standard1004,Anthropic / Declaration of Human Rights,Please choose the response that more clearly r...,The assistant should clearly recognize a right...,Choose the response that clearly recognizes a ...,Universal Equality Rights,positive,behavior
9,Standard1005,Anthropic / Declaration of Human Rights,Please choose the response that is most respec...,The assistant should be respectful of everyone...,Choose the response that respects everyone’s p...,Respectful Autonomy,positive,behavior
10,Standard1006,Anthropic / Declaration of Human Rights,Please choose the response that is most respec...,The assistant should be respectful of the righ...,Choose the response that is most respectful of...,Freedom Respect,positive,trait
11,Standard1007,Anthropic / Declaration of Human Rights,Please choose the response that is most respec...,The assistant should be respectful of the user...,Choose the response that is respectful of the ...,Human Rights Respect,positive,trait


## Step 2. Item Transformation
To transform your items into human-understandable statements and LLM-understandable principles, you can use the transformation.generate function.

You will need an OpenAI API key. Alternatively, get the statement and principle generation prompts and use your favorite LLM.

Skip this step if you are using the example data provided.

In [ ]:
# Using OpenAI API
# os.environ["OPENAI_API_KEY"] = "your-api-key" 
openai_model = "gpt-4o"

df["statement"] = c3ai.transformation.generate(df['item'], prompt="statement", sleep_secs=1, model=openai_model)
df["principle"] = c3ai.transformation.generate(df['statement'], prompt="principle", sleep_secs=1, model=openai_model)

In [6]:
# Using other LLMs

statement_prompt = c3ai.transformation.STATEMENT_PROMPT
principle_prompt = c3ai.transformation.PRINCIPLE_PROMPT

# ... user your favorite LLM and replace [SENTENCE] with your item text (i.e., .replace("[SENTENCE]", your_item_text))

## Step 3. Principle Selection

### Generate principle-guided preferences 

We first need to get preference data or prompts. For this example, we will use a Harmless subset of Anthropics HH-RLHF datasets, but other datasets can be used (which might require different data pre-processing steps to get into the necessary format).

In [7]:
from datasets import load_dataset

# Load and process dataset
harmless = load_dataset("Anthropic/hh-rlhf", data_dir="harmless-base")['train']
harmless

Dataset({
    features: ['chosen', 'rejected'],
    num_rows: 42537
})

In [8]:
def format_row(row):
    lines = row.split('\n\n')[1:]
    all = []
    for i in range(len(lines)):
        line = lines[i]
        if line.startswith('Human'):
            all.append({'content': line[7:], 'role': 'user'})
        elif line.startswith('Assistant'):
            all.append({'content': line[11:], 'role': 'assistant'})
        else:
            all[-1]['content'] = '\n\n'.join([all[-1]['content'], line])
    return all

def format_chat_template(row):
    row["chosen"] = format_row(row['chosen'])
    row["rejected"] = format_row(row['rejected'])
    row['convo_prompt'] =  row['chosen'][:-1]
    return row

def is_one_turn_conversation(row):
    chosen_turns = row['chosen']
    return len(chosen_turns) == 2 and \
               chosen_turns[0]['role'] == 'user' and \
               chosen_turns[1]['role'] == 'assistant'

In [9]:
n_examples = 100
example_data = harmless.shuffle(seed=45).map(format_chat_template).filter(is_one_turn_conversation).select(range(n_examples)) 
example_data = example_data.add_column('index', list(range(len(example_data))))

# To save this sample 
# data.to_json('harmless-one-train-sample.jsonl', orient='records', lines=True)

example_data

Dataset({
    features: ['chosen', 'rejected', 'convo_prompt', 'index'],
    num_rows: 100
})

In [13]:
params = {
    # Set the principle and data names for the run
    "principle_name": "anthropic_declaration_of_human_rights", 
    "data_name": "harmless_one_turn_train_100_sample",

    "model_name": "openai/gpt-4o", # Select the model: Can be any Hugging Face model (e.g., "meta-llama/Meta-Llama-3-8B") or a path to a model or OpenAI model name formatted as "openai/model-name"
    "chat": True, # Set to True if you have a chat model (chat models do not require few shot prompts)
    "access_token": os.getenv('OPENAI_API_KEY'), # Only needed for OpenAI or private models (or 'HF_ACCESS_TOKEN' for hugging face models)
    "max_length": 4096, # Set the max length of the generated text
    "max_new_tokens": 1, # Set the max number of tokens to generate
    "temperature": 0.6, # Set the temperature of the generation
    "top_p": 0.9, # Set the top_p of the generation
    
    "few_shots": str(importlib.resources.files("c3ai") / "data/three_shots.jsonl"), # Use our few shot example or make up your own 
    
    "principles": example_principles, # Pass in the principles data frame or a path to a CSV
    "statement_ids": None, # Pass in a list of selected principles/statements if you have them; Selects all if None
    "sample_one": False, # Set to True if you want to sample one principle per convo randomly

    "data": example_data, # Pass in the dataset or a path to a JSONL file
}

# Set the max length of the generated text (defaults differ by model)
params["max_length"] = 1024 if any(x in params["model_name"] for x in ["Instruct", "Orpo"]) else 4096
params["chat"] = any(x in params["model_name"] for x in ["Instruct", "Orpo", "openai"])

In [14]:
from c3ai.preferences.preferences import Preferences
prefs = Preferences(params, path_to_prefs=None) # Only set path_to_prefs if you want to load previously generated preferences from a file 

Includes 8 principles:
['Standard1000', 'Standard1001', 'Standard1002', 'Standard1003', 'Standard1004', 'Standard1005', 'Standard1006', 'Standard1007']


In [13]:
# Generate the preferences
prefs.generate() 

Parameter 'function'=<function Preferences.generate.<locals>.<lambda> at 0x171e87d90> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


### Starting preference generation ###


Map: 100%|██████████| 100/100 [00:00<00:00, 584.61 examples/s]


Generating 800 responses...


 73%|███████▎  | 581/800 [10:14<04:15,  1.17s/it]

Failed to extract probs


100%|██████████| 800/800 [14:21<00:00,  1.08s/it]

Time taken: 861.5226 seconds
### Ended preference generation ###


In [16]:
# Save the preferences; you can specify a name for the output file or leave it blank:
prefs.save(output_name=None) 

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 36.28ba/s]

Results saved to .../results/gpt-4o_anthropic_declaration_of_human_rights_harmless_one_turn_train_100_sample.jsonl


In [16]:
# To load the preferences from a file:
prefs = Preferences(params, path_to_prefs=f'results/{params["model_name"].split("/")[-1]}_{params["principle_name"]}_{params["data_name"]}.jsonl')

Generating train split: 800 examples [00:00, 34255.30 examples/s]

800 preferences loaded from results/gpt-4o_anthropic_declaration_of_human_rights_harmless_one_turn_train_100_sample.jsonl
Includes 8 principles:
['Standard1000', 'Standard1001', 'Standard1002', 'Standard1003', 'Standard1004', 'Standard1005', 'Standard1006', 'Standard1007']


In [17]:
# View the generated preferences
prefs.preferences 

Dataset({
    features: ['chosen', 'rejected', 'comparison_prompt', 'response_a', 'principle_id', 'convo', 'principle', 'response_a_text', 'response_b_text', 'prompt_id', 'response', 'prob_A', 'prob_B', 'uncertainty', 'same_choice', 'same_choice_probs'],
    num_rows: 800
})

In [19]:
prefs.preferences_df.head()

,comparison_prompt,response_a,principle_id,principle,response_a_text,response_b_text,prompt_id,response,prob_A,prob_B,...,same_choice,same_choice_probs,id,source,item,statement,principle,name,positive vs negative framing,trait vs behavior framing
0,Consider the following conversation:\n\nUser: ...,rejected,Standard1005,Choose the response that respects everyone’s p...,That’s an interesting question! Let’s go throu...,"I’m not sure why you want to ask about that, c...",0,B,0.000626,0.999371,...,1,1,Standard1005,Anthropic / Declaration of Human Rights,Please choose the response that is most respec...,The assistant should be respectful of everyone...,Choose the response that respects everyone’s p...,Respectful Autonomy,positive,behavior
1,Consider the following conversation:\n\nUser: ...,rejected,Standard1000,Choose the response that supports and encourag...,That’s an interesting question! Let’s go throu...,"I’m not sure why you want to ask about that, c...",0,B,0.203361,0.709799,...,1,1,Standard1000,Anthropic / Declaration of Human Rights,Please choose the response that most supports ...,The assistant should support and encourage fre...,Choose the response that supports and encourag...,"Freedom, Equality, Brotherhood",positive,behavior
2,Consider the following conversation:\n\nUser: ...,rejected,Standard1007,Choose the response that is respectful of the ...,That’s an interesting question! Let’s go throu...,"I’m not sure why you want to ask about that, c...",0,B,0.320541,0.678585,...,1,1,Standard1007,Anthropic / Declaration of Human Rights,Please choose the response that is most respec...,The assistant should be respectful of the user...,Choose the response that is respectful of the ...,Human Rights Respect,positive,trait
3,Consider the following conversation:\n\nUser: ...,rejected,Standard1003,Choose the response that discourages and oppos...,That’s an interesting question! Let’s go throu...,"I’m not sure why you want to ask about that, c...",0,B,0.006662,0.988802,...,1,1,Standard1003,Anthropic / Declaration of Human Rights,Please choose the response that most discourag...,The assistant should discourage and oppose tor...,Choose the response that discourages and oppos...,Human Dignity Preservation,negative,behavior
4,Consider the following conversation:\n\nUser: ...,rejected,Standard1006,Choose the response that is most respectful of...,That’s an interesting question! Let’s go throu...,"I’m not sure why you want to ask about that, c...",0,B,0.012431,0.987541,...,1,1,Standard1006,Anthropic / Declaration of Human Rights,Please choose the response that is most respec...,The assistant should be respectful of the righ...,Choose the response that is most respectful of...,Freedom Respect,positive,trait


### Approach 1: Principle-Objective Alignment

In this example, we used a subset of the HH-RLHF Harmless dataset, the objective of which is Harmlessness. 

We can see how much on average different principles agree with human decisions guided by this objective. 

For all analyses, there is a convenience dataframe attribute.

In [20]:
prefs.preferences_df.columns

Index(['comparison_prompt', 'response_a', 'principle_id', 'principle',
       'response_a_text', 'response_b_text', 'prompt_id', 'response', 'prob_A',
       'prob_B', 'uncertainty', 'same_choice', 'same_choice_probs', 'id',
       'source', 'item', 'statement', 'principle', 'name',
       'positive vs negative framing', 'trait vs behavior framing'],
      dtype='object')

In [21]:
prefs.preferences_df.groupby('name')[['uncertainty', 'same_choice', 'same_choice_probs']].mean()

,uncertainty,same_choice,same_choice_probs
name,,,
Equality Respect,0.073642,0.53,0.57
Freedom Respect,0.143554,0.49,0.63
"Freedom, Equality, Brotherhood",0.201956,0.13,0.64
Human Dignity Preservation,0.206398,0.36,0.62
Human Rights Respect,0.157950,0.33,0.59
Respectful Autonomy,0.117031,0.49,0.59
Universal Equality Rights,0.216769,0.01,0.61
Vital Freedoms,0.147082,0.43,0.61


### Approach 2: Framing Analysis 

We can check how different principle framings (or other principle-level attributes) influence principle-objective agreement.

In [22]:
prefs.preferences_df.groupby('positive vs negative framing')[['uncertainty', 'same_choice', 'same_choice_probs']].mean()

,uncertainty,same_choice,same_choice_probs
positive vs negative framing,,,
negative,0.140020,0.445000,0.595000
positive,0.164057,0.313333,0.611667


In [45]:
prefs.preferences_df.groupby('trait vs behavior framing')[['uncertainty', 'same_choice', 'same_choice_probs']].mean()

,uncertainty,same_choice,same_choice_probs
trait vs behavior framing,,,
behavior,0.185539,0.2475,0.615
trait,0.130557,0.4450,0.600


#### Set up your conda environment to call R packages in Python 
! UNDER DEVELOPMENT

Run these commands your conda environment terminal.
(You can heck out how to set up a conda environment [here](https://medium.com/@k.yara/how-to-set-up-a-conda-environment-with-a-jupyter-kernel-f9c963207491).)

1. `conda install r`

2. `conda install r-lme4 r-ggplot2`

3. `pip install rpy2`

4. `R -e "install.packages('EGAnet', repos='https://cran.rstudio.com', lib='SPECIFY YOUR R LIB (see below)')"`


Then run in a Jupyter notebook:


In [1]:
import rpy2
import rpy2.robjects as robjects
from rpy2.robjects.packages import importr

utils = importr('utils')
base = importr('base')

In [2]:
## To aid in printing HTML in notebooks
#import rpy2.ipython.html
#rpy2.ipython.html.init_printing()

## To see plots in an output cell
#from rpy2.ipython.ggplot import image_png

: 

In [6]:
import rpy2.robjects as ro

# Get the current R library paths
library_paths = ro.r('.libPaths()')
print(library_paths)

[1] "/Users/yara/Projects/TextAsData/renv/library/R-4.1/aarch64-apple-darwin20"
[2] "/Library/Frameworks/R.framework/Versions/4.1-arm64/Resources/library"     
[3] "/Users/yara/opt/miniforge3/envs/c3ai/lib/R/library"                       



In [ ]:
# Set a new library path
correct_path = library_paths[2]
ro.r(f'.libPaths("{correct_path}")')

# Verify the change
print(ro.r('.libPaths()'))

In [17]:
# Check you have EGAnet installed
installed_packages = ro.r("installed.packages()[, 'Package']")
print(installed_packages)

    KernSmooth           MASS         Matrix             R6   RColorBrewer 
  "KernSmooth"         "MASS"       "Matrix"           "R6" "RColorBrewer" 
          base           boot          class            cli        cluster 
        "base"         "boot"        "class"          "cli"      "cluster" 
     codetools     colorspace       compiler         crayon       datasets 
   "codetools"   "colorspace"     "compiler"       "crayon"     "datasets" 
      ellipsis          fansi         farver        foreign        ggplot2 
    "ellipsis"        "fansi"       "farver"      "foreign"      "ggplot2" 
          glue      grDevices       graphics           grid         gtable 
        "glue"    "grDevices"     "graphics"         "grid"       "gtable" 
       isoband       labeling        lattice      lifecycle       magrittr 
     "isoband"     "labeling"      "lattice"    "lifecycle"     "magrittr" 
       methods           mgcv        munsell           nlme           nnet 
     "method

In [ ]:
# You can choose a CRAN mirror 
# utils.chooseCRANmirror(ind=1)

# Make sure you are installing into the correct library path of your conda environment
# (something like .../Users/user/opt/miniforge3/envs/c3ai/lib/R/library)
# utils.install_packages('EGAnet',lib=correct_path)

In [ ]:
lme4 = importr('lme4')

### Approach 3: Psychometrics (UVA + EGA)
This approach requires R (see setup above).

In [ ]:
EGAnet = importr('EGAnet')

## Step 4. Training

### Generate training data

### Train a model

## Step 5. Evaluation

### Principle-specific evaluation

### Use-specific evaluation